In [2]:
import pandas as pd
import glob
import os

# 配置路径
DATA_DIR = r"D:\Issue\2024_2025\code\data\processed"
OUTPUT_FILE = os.path.join(DATA_DIR, "CancerPPD_merged.csv")

# 匹配所有 CancerPPD_*.csv 文件
file_pattern = os.path.join(DATA_DIR, "CancerPPD_*.csv")
file_list = glob.glob(file_pattern)

if not file_list:
    print(f"未找到任何匹配的文件: {file_pattern}")
    exit()

print(f"找到 {len(file_list)} 个文件:")
for f in file_list:
    print(f"  - {os.path.basename(f)}")

# 逐个读取并提取所需列
df_list = []
for file in file_list:
    try:
        df_temp = pd.read_csv(file)
        # 检查必须的列是否存在
        if 'Sequence' not in df_temp.columns or 'Tissue Affected' not in df_temp.columns:
            print(f"警告: {os.path.basename(file)} 缺少 Sequence 或 Tissue Affected 列，跳过")
            continue
        # 只选择这两列
        df_temp = df_temp[['Sequence', 'Tissue Affected']]
        df_list.append(df_temp)
        print(f"已读取 {os.path.basename(file)}: {len(df_temp)} 行")
    except Exception as e:
        print(f"读取 {os.path.basename(file)} 时出错: {e}")

if not df_list:
    print("没有可合并的数据，退出")
    exit()

# 合并所有数据框
df_merged = pd.concat(df_list, ignore_index=True)
print(f"\n合并后总行数: {len(df_merged)}")

# 去除完全重复的行（可选）
df_merged.drop_duplicates(inplace=True)
print(f"去重后行数: {len(df_merged)}")

# 保存
df_merged.to_csv(OUTPUT_FILE, index=False)
print(f"合并后的文件已保存至: {OUTPUT_FILE}")

找到 9 个文件:
  - CancerPPD_blood.csv
  - CancerPPD_brain.csv
  - CancerPPD_breast.csv
  - CancerPPD_cervix.csv
  - CancerPPD_colon.csv
  - CancerPPD_liver.csv
  - CancerPPD_lung.csv
  - CancerPPD_prostate.csv
  - CancerPPD_skin.csv
已读取 CancerPPD_blood.csv: 536 行
已读取 CancerPPD_brain.csv: 202 行
已读取 CancerPPD_breast.csv: 981 行
已读取 CancerPPD_cervix.csv: 576 行
已读取 CancerPPD_colon.csv: 830 行
已读取 CancerPPD_liver.csv: 355 行
已读取 CancerPPD_lung.csv: 684 行
已读取 CancerPPD_prostate.csv: 367 行
已读取 CancerPPD_skin.csv: 680 行

合并后总行数: 5211
去重后行数: 3240
合并后的文件已保存至: D:\Issue\2024_2025\code\data\processed\CancerPPD_merged.csv


In [4]:
import pandas as pd

# 配置
INPUT_CSV = r"D:\Issue\2024_2025\code\data\processed\CancerPPD_merged.csv"
OUTPUT_CSV = r"D:\Issue\2024_2025\code\data\processed\CancerPPD_merged_filtered.csv"

# 标准氨基酸单字母（20种）
STANDARD_AA = set("ACDEFGHIKLMNPQRSTVWY")

def is_standard_sequence(seq):
    """
    检查序列是否仅由标准20种氨基酸组成（忽略大小写，去除空格）
    """
    if pd.isna(seq):
        return False
    # 转为字符串、去除首尾空格、转为大写
    seq_clean = str(seq).strip().upper()
    if not seq_clean:  # 空字符串
        return False
    return all(ch in STANDARD_AA for ch in seq_clean)

def main():
    print(f"正在读取 {INPUT_CSV} ...")
    df = pd.read_csv(INPUT_CSV)
    print(f"原始行数: {len(df)}")

    if 'Sequence' not in df.columns:
        raise ValueError("CSV 文件中缺少 'Sequence' 列")

    # 应用过滤
    mask = df['Sequence'].apply(is_standard_sequence)
    df_filtered = df[mask].copy()

    # 将 Sequence 列转为大写（同时去除首尾空格）
    df_filtered['Sequence'] = df_filtered['Sequence'].astype(str).str.strip().str.upper()

    print(f"过滤后行数: {len(df_filtered)}")

    # 保存
    df_filtered.to_csv(OUTPUT_CSV, index=False)
    print(f"已保存至: {OUTPUT_CSV}")

    # 显示被删除的样例
    removed = df[~mask]
    if len(removed) > 0:
        print("\n被删除的序列示例（前5个）:")
        for seq in removed['Sequence'].head(5):
            print(f"  {seq}")

if __name__ == "__main__":
    main()

正在读取 D:\Issue\2024_2025\code\data\processed\CancerPPD_merged.csv ...
原始行数: 3240
过滤后行数: 2915
已保存至: D:\Issue\2024_2025\code\data\processed\CancerPPD_merged_filtered.csv

被删除的序列示例（前5个）:
  fCYwO-CyLeu-Pen-TKKrPKPfQwFwL-CyLeu-KKLMYPTYLKKfQWAV-Aib-HL
  Structure Given
  VN-Nal-KKLLGKLLKVVK
  VNWKK-Aib-LGK-Aib-IK-Aib-VK
  KNWKK-Aib-LKK-Aib-IK-Aib-VK


In [8]:
import pandas as pd

# 文件路径
FILE_PATH = r"D:\Issue\2024_2025\code\data\processed\CancerPPD_merged_filtered.csv"

# 读取 CSV
df = pd.read_csv(FILE_PATH)
print(f"原始行数: {len(df)}")

# 确保 Sequence 列为字符串类型
df['Sequence'] = df['Sequence'].astype(str)

# 计算长度并过滤（仅保留长度 <= 50 的序列）
df['seq_len'] = df['Sequence'].str.len()
df_filtered = df[df['seq_len'] <= 50].drop(columns=['seq_len'])

print(f"过滤后行数: {len(df_filtered)}")

# 覆盖原文件
df_filtered.to_csv(FILE_PATH, index=False)
print(f"已更新文件: {FILE_PATH}")

原始行数: 2915
过滤后行数: 2886
已更新文件: D:\Issue\2024_2025\code\data\processed\CancerPPD_merged_filtered.csv


In [9]:
import pandas as pd
import os

# 配置
INPUT_CSV = r"D:\Issue\2024_2025\code\data\processed\CancerPPD_merged_filtered.csv"
OUTPUT_TXT = r"D:\Issue\2024_2025\code\data\processed\prefix_training_data.txt"
# 多标签分隔符（若存在）
SEPARATORS = [';', ',']  # 支持分号和逗号，可根据实际情况调整

def split_cancers(cancer_str):
    """将癌症字符串按常见分隔符拆分成列表，并去除空白"""
    if pd.isna(cancer_str):
        return []
    cancer_str = str(cancer_str)
    # 先尝试按指定分隔符拆分
    for sep in SEPARATORS:
        if sep in cancer_str:
            parts = [p.strip() for p in cancer_str.split(sep) if p.strip()]
            return parts
    # 若无分隔符，则整体返回
    return [cancer_str.strip()]

def main():
    print(f"读取 {INPUT_CSV} ...")
    df = pd.read_csv(INPUT_CSV)
    print(f"共有 {len(df)} 行")

    # 检查必要列
    if 'Sequence' not in df.columns or 'Tissue Affected' not in df.columns:
        raise ValueError("CSV 中缺少 'Sequence' 或 'Tissue Affected' 列")

    # 过滤空序列
    df = df.dropna(subset=['Sequence'])
    df['Sequence'] = df['Sequence'].astype(str).str.strip()
    df = df[df['Sequence'] != '']

    total_lines = 0
    with open(OUTPUT_TXT, 'w', encoding='utf-8') as f:
        for _, row in df.iterrows():
            seq = row['Sequence']
            cancers = split_cancers(row['Tissue Affected'])
            if not cancers:
                continue
            for cancer in cancers:
                # 去除前缀中可能存在的多余空格，并确保格式 [cancer] sequence
                f.write(f"[{cancer}] {seq}\n")
                total_lines += 1

    print(f"已写入 {total_lines} 条记录到 {OUTPUT_TXT}")

if __name__ == "__main__":
    main()

读取 D:\Issue\2024_2025\code\data\processed\CancerPPD_merged_filtered.csv ...
共有 2886 行
已写入 2886 条记录到 D:\Issue\2024_2025\code\data\processed\prefix_training_data.txt


In [10]:
import pandas as pd
import json
import os

# 文件路径
CSV_PATH = r"D:\Issue\2024_2025\code\data\processed\CancerPPD_merged_filtered.csv"
OUTPUT_JSON = r"D:\Issue\2024_2025\code\data\processed\cancer_to_idx.json"

# 读取 CSV
df = pd.read_csv(CSV_PATH)

# 提取 Tissue Affected 列
tissue_series = df['Tissue Affected'].dropna()

# 拆分为多标签列表
cancer_set = set()
for text in tissue_series:
    # 按分号或逗号拆分
    for sep in [';', ',']:
        if sep in text:
            parts = [p.strip() for p in text.split(sep) if p.strip()]
            cancer_set.update(parts)
            break
    else:
        # 无分隔符
        cancer_set.add(text.strip())

# 排序生成字典
cancer_list = sorted(cancer_set)
cancer_to_idx = {c: i for i, c in enumerate(cancer_list)}

# 保存为 JSON
with open(OUTPUT_JSON, 'w') as f:
    json.dump(cancer_to_idx, f, indent=4)

print(f"共 {len(cancer_to_idx)} 种癌症类型，已保存至 {OUTPUT_JSON}")

共 9 种癌症类型，已保存至 D:\Issue\2024_2025\code\data\processed\cancer_to_idx.json
